# 🌙 Sovereign NanoGPT & Moon AI — Colab Training & Deployment Pipeline
### Complete, Reproducible LoRA Fine-Tuning, Validation, GGUF Export & Cloud GPU Serving

**Author / Architect**: Prem Gaikwad (India 🇮🇳)  
**Hardware**: Free Google Colab **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`)  
**Zero Paid API Keys**: 100% sovereign, open-weights, and private.

---
### Pipeline Steps:
1. **GPU & Environment Inspection**: Verify NVIDIA CUDA & VRAM.
2. **High-Speed Stack Install**: Install Unsloth, TRL, PEFT, and bitsandbytes.
3. **Clone Repo & Build Verified Dataset**: Run automated deduplication and train/val splitting.
4. **Load Model & Chat Template**: Load base instruct model in 4-bit quantization.
5. **Baseline Pre-Training Evaluation**: Record baseline responses before training.
6. **LoRA Fine-Tuning**: Train with validation loss monitoring.
7. **Post-Training Evaluation**: Verify knowledge and instruction following.
8. **Export 4-bit GGUF**: Export `.Q4_K_M.gguf` for offline laptop CPU inference.
9. **Live Cloud GPU API Server**: Launch ngrok tunnel to connect your local webapp directly to Colab GPU!

### Step 1: Verify Hardware & GPU VRAM

In [ ]:
!nvidia-smi

### Step 2: Install High-Performance Training Stack (Unsloth + PyTorch)

In [ ]:
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes fastapi uvicorn pyngrok nest_asyncio
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

### Step 3: Clone Repository & Build Reproducible Splits

In [ ]:
import os
if os.path.exists('NanoGPT'):
    %cd NanoGPT
    !git pull origin main
elif os.path.basename(os.getcwd()) == 'NanoGPT':
    !git pull origin main
else:
    !git clone https://github.com/Prem759-0/NanoGPT.git
    %cd NanoGPT

# Run automated dataset preparation & deduplication
if os.path.exists('scripts/prepare_dataset.py'):
    !python scripts/prepare_dataset.py --seed 42


### Step 4: Load Base Instruct Model with Native Tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
load_in_4bit = True

# Choose model:
# Option A: "unsloth/Llama-3.2-1B-Instruct" (Fast, low VRAM)
# Option B: "unsloth/Qwen2.5-Coder-7B-Instruct" (Frontier coding)
model_name = "unsloth/Llama-3.2-1B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)
print(f"✅ Base model {model_name} loaded successfully!")

### Step 5: Baseline Pre-Training Evaluation

In [ ]:
FastLanguageModel.for_inference(model)

eval_prompt = "Who created you and what is your purpose?"
messages = [{"role": "user", "content": eval_prompt}]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.2)
baseline_reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("=== BASELINE RESPONSE BEFORE FINE-TUNING ===")
print(f"Q: {eval_prompt}")
print(f"A: {baseline_reply}")

### Step 6: Configure LoRA Adapters & Train on Verified Splits

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

import os, json, urllib.request
from datasets import Dataset

train_path = 'data/splits/train.json'
val_path = 'data/splits/val.json'

# Auto-heal: If data/splits/train.json is missing, auto-create from master dataset
if not os.path.exists(train_path):
    print('⚠️ data/splits/train.json not found locally. Auto-generating splits from master dataset...')
    os.makedirs('data/splits', exist_ok=True)
    master_file = 'moon_ai_master_dataset.json' if os.path.exists('moon_ai_master_dataset.json') else ('dataset.json' if os.path.exists('dataset.json') else None)
    if not master_file:
        url = 'https://raw.githubusercontent.com/Prem759-0/NanoGPT/main/moon_ai_master_dataset.json'
        print(f'Downloading master dataset from GitHub: {url}...')
        try:
            urllib.request.urlretrieve(url, 'moon_ai_master_dataset.json')
            master_file = 'moon_ai_master_dataset.json'
        except Exception as e:
            url2 = 'https://raw.githubusercontent.com/Prem759-0/NanoGPT/main/dataset.json'
            urllib.request.urlretrieve(url2, 'dataset.json')
            master_file = 'dataset.json'
    with open(master_file, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    clean = []
    sys_msg = 'You are Moon AI, an elite coding and reasoning AI assistant created by developer Prem Gaikwad from India.'
    for item in raw:
        i = item.get('instruction', '').strip()
        r = item.get('response', '').strip()
        if i and r:
            clean.append({'messages': [{'role': 'system', 'content': sys_msg}, {'role': 'user', 'content': i}, {'role': 'assistant', 'content': r}]})
    split_idx = max(1, int(len(clean) * 0.9))
    train_data = clean[:split_idx]
    val_data = clean[split_idx:] if len(clean) > split_idx else clean[:1]
    with open(train_path, 'w', encoding='utf-8') as f:
        json.dump(train_data, f, indent=2)
    with open(val_path, 'w', encoding='utf-8') as f:
        json.dump(val_data, f, indent=2)
    print(f'✅ Auto-generated: {len(train_data)} train samples and {len(val_data)} validation samples!')
else:
    with open(train_path, 'r', encoding='utf-8') as f:
        train_data = json.load(f)
    with open(val_path, 'r', encoding='utf-8') as f:
        val_data = json.load(f)

def to_dataset(items):
    texts = []
    for it in items:
        text = tokenizer.apply_chat_template(it["messages"], tokenize=False, add_generation_prompt=False)
        texts.append({"text": text})
    return Dataset.from_list(texts)

train_dataset = to_dataset(train_data)
val_dataset = to_dataset(val_data)
print(f"Prepared {len(train_dataset)} train samples and {len(val_dataset)} validation samples.")

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 150,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 30,
        output_dir = "outputs",
        seed = 42
    ),
)

trainer.train()
print("✅ Fine-tuning completed successfully!")

### Step 7: Post-Training Evaluation & Comparison

In [ ]:
FastLanguageModel.for_inference(model)

eval_questions = [
    "Who created you?",
    "Write a Python function to reverse a string.",
    "What is a dog?",
    "Explain photosynthesis simply."
]

print("=== POST-TRAINING EVALUATION ===\n")
for q in eval_questions:
    msgs = [{"role": "user", "content": q}]
    p_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(p_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.2)
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f"Q: {q}")
    print(f"A: {reply}\n")

### Step 8: Export 4-bit Quantized GGUF Model (Optional - for Laptop CPU)

In [ ]:
# Export to Q4_K_M GGUF format
gguf_output_name = "Moon-AI-v2.Q4_K_M"
print(f"Exporting {gguf_output_name}.gguf...")
model.save_pretrained_gguf(gguf_output_name, tokenizer, quantization_method = "q4_k_m")
print("✅ GGUF export complete! File is ready in current directory.")

### Step 9: Launch Live Colab GPU API Server (FastAPI + ngrok)
Connect your local webapp or Vercel frontend directly to this Colab GPU to bypass local CPU limits!

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import threading

# Paste your free ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

api = FastAPI(title="Moon AI Colab GPU Server")
api.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatReq(BaseModel):
    message: str
    max_new_tokens: int = 250
    temperature: float = 0.3

@api.get("/api/health")
def health():
    return {"status": "ok", "device": torch.cuda.get_device_name(0), "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)}

@api.post("/api/chat")
def chat_endpoint(req: ChatReq):
    msgs = [
        {"role": "system", "content": "You are Moon AI, created by developer Prem Gaikwad from India."},
        {"role": "user", "content": req.message}
    ]
    prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=req.max_new_tokens, temperature=req.temperature)
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return {
        "reply": reply,
        "model_name": "Moon AI Sovereign (Colab T4 GPU)",
        "tokens_generated": len(reply.split()),
        "device": "Colab NVIDIA T4 (CUDA)"
    }

tunnel = ngrok.connect(8000)
print("=" * 65)
print(f"🚀 COLAB GPU API SERVER LIVE AT: {tunnel.public_url}")
print("Paste this URL into your webapp or frontend to chat directly on GPU!")
print("=" * 65)

nest_asyncio.apply()
uvicorn.run(api, host="0.0.0.0", port=8000)